In [ ]:
import pandas as pd
from models import Ride, ride_serializer, ride_from_row
import time
from kafka import KafkaProducer

In [ ]:
cols = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance",
    "tip_amount",
    "total_amount"
]
df = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet", columns=cols)

In [ ]:
df.iloc[0]

In [ ]:
row = df.iloc[0]
ride = ride_from_row(row)
ride


In [ ]:
producer = KafkaProducer(
    bootstrap_servers='localhost:9092', 
    value_serializer=ride_serializer
    )

In [ ]:
df.count()

In [ ]:
topic_name = 'green-trips'

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')